# 01b. labeled 샷 단위 정리 + 가동 구간(Run) 나누기

분석 전에 labeled 데이터를 **샷 단위**로 정리하고 **가동 구간(run)** 을 붙인 기본 테이블을 만듭니다. 이후 분석(02, 03, 모델링)은 이 테이블을 공통으로 씁니다.

**1. 샷 단위 정리**
- 같은 `TimeStamp`의 LH·RH 두 행 = 한 번의 사출(샷)에서 나온 두 제품. 공정값이 같으므로 **한 행만 남김**
- 라벨: **LH·RH 둘 다 정상이면 정상(0), 하나라도 불량이면 불량(1)**
- 어느 캐비티가 불량이었는지(`LH_fail`, `RH_fail`), 불량 유형(`Reason`)은 **분석용으로만** 따로 보관

**2. 가동 구간(run) 나누기**
- 날짜로 나누면 자정을 넘는 가동이 쪼개지고(CN7 10-19 23:24 ~ 10-20 06:20), 하루 안의 긴 정지는 못 잡음
- 직전 샷과의 간격이 `RUN_GAP_MIN`분보다 길면 **새 가동 구간**으로 나눔. 정상 사이클은 약 60초
- 그보다 짧은 정지(`STOP_GAP_MIN`분 초과)는 같은 구간 안의 **짧은 정지**로 표시 → 재시작 직후 샷을 따로 볼 수 있게

**산출물**: `linked/shots_labeled.csv` (샷 1행 = 1샷)

## 1. 설정

In [ ]:
LINKED_DIR = './linked'          # 01 노트북 산출물
DATA_DIR = '../data'             # 대회 2차 파일 (moldset_labeled_*.csv) — 표준화 값 붙이기용
OUT_PATH = './linked/shots_labeled.csv'

RUN_GAP_MIN = 30                 # 이 시간(분)보다 길게 멈추면 새 가동 구간
STOP_GAP_MIN = 5                 # 이 시간(분)보다 길게 멈추면 '짧은 정지' 표시 (RUN_GAP_MIN 이하)

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib import font_manager
_av = {f.name for f in font_manager.fontManager.ttflist}
for _f in ['Malgun Gothic', 'AppleGothic', 'NanumGothic', 'Noto Sans CJK KR', 'Noto Sans CJK JP']:
    if _f in _av:
        mpl.rcParams['font.family'] = _f; break
mpl.rcParams.update({'axes.unicode_minus': False, 'axes.spines.top': False, 'axes.spines.right': False})
pd.set_option('display.max_columns', 60); pd.set_option('display.width', 200)
BLUE, LIGHT, RED, GRAY = '#2a78d6', '#86b6ef', '#d03b3b', '#c3c2b7'

FEATURES = ['Injection_Time', 'Filling_Time', 'Plasticizing_Time', 'Cycle_Time', 'Clamp_Close_Time',
            'Cushion_Position', 'Plasticizing_Position', 'Clamp_Open_Position', 'Max_Injection_Speed',
            'Max_Screw_RPM', 'Average_Screw_RPM', 'Max_Injection_Pressure', 'Max_Switch_Over_Pressure',
            'Max_Back_Pressure', 'Average_Back_Pressure', 'Barrel_Temperature_1', 'Barrel_Temperature_2',
            'Barrel_Temperature_3', 'Barrel_Temperature_4', 'Barrel_Temperature_5', 'Barrel_Temperature_6',
            'Hopper_Temperature', 'Mold_Temperature_3', 'Mold_Temperature_4']
RAW = [f'{c}_raw' for c in FEATURES]
PRODUCTS = ['cn7', 'rg3']

## 2. LH·RH 두 행 구조 확인

샷(`product` + `TimeStamp`)마다 행이 몇 개인지, 두 행의 공정값이 정말 같은지 확인합니다.

In [ ]:
L = pd.read_csv(os.path.join(LINKED_DIR, 'linked_labeled.csv'), parse_dates=['TimeStamp', 'plan_date'])
L['shot_key'] = L['product'] + '_' + L['TimeStamp'].dt.strftime('%Y%m%d%H%M%S')

g = L.groupby('shot_key')
struct = pd.DataFrame({
    'product': g['product'].first(),
    '행 수': g.size(),
    '캐비티': g['cavity'].agg(lambda s: '+'.join(sorted(s))),
    '공정값이 다른 컬럼 수': g[RAW].nunique().gt(1).sum(axis=1),
})
print('샷별 행 수 / 캐비티 구성')
display(struct.groupby(['product', '행 수', '캐비티']).size().rename('샷 수').to_frame())
n_diff = int((struct['공정값이 다른 컬럼 수'] > 0).sum())
print(f'LH·RH 공정값이 하나라도 다른 샷: {n_diff}개')
assert n_diff == 0, '공정값이 다른 샷이 있음 → 한 행만 남기면 정보 손실. 확인 필요'

single = struct[struct['행 수'] == 1]
if len(single):
    print('\n한 행만 있는 샷 (그대로 1샷으로 사용):')
    display(L[L['shot_key'].isin(single.index)][['product', 'TimeStamp', 'cavity', 'PassOrFail', 'Reason', 'in_secondary']])

## 3. 샷 단위 테이블 만들기

| 컬럼 | 의미 | 모델 입력 |
|---|---|---|
| `PassOrFail` | **샷 라벨**: LH·RH 중 하나라도 불량이면 1 | 타깃 |
| `LH_fail`, `RH_fail` | 캐비티별 불량 여부 (행이 없으면 NaN) | ❌ 분석용 |
| `fail_pattern` | 정상 / LH만 / RH만 / 양쪽 | ❌ 분석용 |
| `Reason_LH`, `Reason_RH` | 불량 유형 (가스·미성형 등) | ❌ **입력 금지** (검사 후 정보) |
| `n_rows`, `cavities` | 원래 행 수, 캐비티 구성 | ❌ |
| `sec_index_LH`, `sec_index_RH` | 대회 2차 파일의 행 번호(`Unnamed: 0`) | ❌ 연결용 |
| `*_raw` | 원래 단위 공정값 24개 | ✅ |
| 공정변수 이름 그대로 (예: `Injection_Time`) | 대회 2차 파일의 표준화 값 | ✅ |

In [ ]:
def per_cavity(df, col, cav):
    return df[df['cavity'] == cav].set_index('shot_key')[col]

first = L.sort_values(['shot_key', 'cavity']).groupby('shot_key').first()
S = first[['product', 'shot_id', 'TimeStamp', 'plan_date', 'in_secondary'] + RAW].copy()
S['n_rows'] = struct['행 수']
S['cavities'] = struct['캐비티']
S['LH_fail'] = per_cavity(L, 'PassOrFail', 'LH')
S['RH_fail'] = per_cavity(L, 'PassOrFail', 'RH')
S['Reason_LH'] = per_cavity(L, 'Reason', 'LH')
S['Reason_RH'] = per_cavity(L, 'Reason', 'RH')
S['sec_index_LH'] = per_cavity(L, 'sec_index', 'LH')
S['sec_index_RH'] = per_cavity(L, 'sec_index', 'RH')

# 샷 라벨: 하나라도 불량이면 1, 둘 다 정상이면 0
S['PassOrFail'] = S[['LH_fail', 'RH_fail']].max(axis=1).astype(int)
lh, rh = S['LH_fail'].fillna(0).astype(int), S['RH_fail'].fillna(0).astype(int)
S['fail_pattern'] = np.select([(lh == 1) & (rh == 1), lh == 1, rh == 1], ['양쪽', 'LH만', 'RH만'], '정상')
S = S.reset_index()

In [ ]:
# 대회 2차 파일의 표준화 값 붙이기 (sec_index로 연결). LH·RH 두 행 값이 같은지도 확인
parts = []
for p in PRODUCTS:
    sec = pd.read_csv(os.path.join(DATA_DIR, f'moldset_labeled_{p}.csv')).set_index('Unnamed: 0')
    s = S[S['product'] == p]
    both = s[s['sec_index_LH'].notna() & s['sec_index_RH'].notna()]
    if len(both):
        a = sec.loc[both['sec_index_LH'].astype(int), FEATURES].values
        b = sec.loc[both['sec_index_RH'].astype(int), FEATURES].values
        print(f'[{p}] 2차 파일에서 LH·RH 표준화 값 최대 차이: {np.abs(a - b).max():.2e}')
    use = s['sec_index_LH'].fillna(s['sec_index_RH'])
    z = pd.DataFrame(np.nan, index=s.index, columns=FEATURES)
    ok = use.notna()
    z.loc[ok, FEATURES] = sec.loc[use[ok].astype(int), FEATURES].values
    parts.append(z)
S = S.join(pd.concat(parts))
print('2차 표준화 값이 없는 샷 (2차 범위 밖):', int(S[FEATURES[0]].isna().sum()))

### 정리 전후 비교

In [ ]:
rows = []
for p in PRODUCTS:
    for scope, cond_L, cond_S in [('2차 범위', L['in_secondary'], S['in_secondary']), ('전체(1차 포함)', True, True)]:
        l = L[(L['product'] == p) & cond_L]
        s = S[(S['product'] == p) & cond_S]
        rows.append({'product': p, '범위': scope, '행(정리 전)': len(l), '불량 행': int(l['PassOrFail'].sum()),
                     '샷(정리 후)': len(s), '불량 샷': int(s['PassOrFail'].sum()),
                     '불량률(샷)': f"{s['PassOrFail'].mean():.1%}",
                     'RH만': int((s['fail_pattern'] == 'RH만').sum()), 'LH만': int((s['fail_pattern'] == 'LH만').sum()),
                     '양쪽': int((s['fail_pattern'] == '양쪽').sum())})
display(pd.DataFrame(rows).set_index(['product', '범위']))

## 4. 가동 구간(run) 나누기

- 제품별로 시간순 정렬 → 직전 샷과의 간격(`prev_gap_sec`)
- 간격 > `RUN_GAP_MIN`분 → 새 구간 (`run_id` = `cn7_R1`, `cn7_R2`, …)
- `STOP_GAP_MIN`분 < 간격 ≤ `RUN_GAP_MIN`분 → 같은 구간 안의 짧은 정지 (`after_stop = True`)

| 추가 컬럼 | 의미 | 모델 입력 |
|---|---|---|
| `run_id`, `run_no` | 가동 구간 | ❌ 분할·그룹용 |
| `shot_seq_in_run` | 구간 시작부터 몇 번째 샷인지 (1부터) | ✅ 후보 |
| `min_from_run_start` | 구간 시작 후 경과 시간(분) | ✅ 후보 |
| `prev_gap_sec` | 직전 샷과의 간격(초). 구간 첫 샷은 NaN | ✅ 후보 |
| `after_stop` | 짧은 정지 직후 첫 샷 | ✅ 후보 |
| `shots_since_stop` | 구간 시작 또는 마지막 짧은 정지 이후 몇 번째 샷인지 | ✅ 후보 |

In [ ]:
S = S.sort_values(['product', 'TimeStamp']).reset_index(drop=True)
S['prev_gap_sec'] = S.groupby('product')['TimeStamp'].diff().dt.total_seconds()
new_run = S['prev_gap_sec'].isna() | (S['prev_gap_sec'] > RUN_GAP_MIN * 60)
S['run_no'] = new_run.groupby(S['product']).cumsum().astype(int)
S['run_id'] = S['product'] + '_R' + S['run_no'].astype(str)
S.loc[new_run, 'prev_gap_sec'] = np.nan

S['shot_seq_in_run'] = S.groupby('run_id').cumcount() + 1
S['min_from_run_start'] = (S['TimeStamp'] - S.groupby('run_id')['TimeStamp'].transform('min')).dt.total_seconds() / 60
S['after_stop'] = S['prev_gap_sec'] > STOP_GAP_MIN * 60
seg = (new_run | S['after_stop']).groupby(S['run_id']).cumsum()
S['shots_since_stop'] = S.groupby([S['run_id'], seg]).cumcount() + 1

# 간격 분포 확인: 정상 사이클(약 60초)과 정지가 잘 나뉘는지
gaps = S['prev_gap_sec'].dropna()
print('샷 간격(초) 분포:', gaps.describe(percentiles=[.5, .9, .99]).round(1).to_dict())
print(f'{STOP_GAP_MIN}분 초과 짧은 정지:', int(S['after_stop'].sum()), '회')

### 가동 구간 요약

In [ ]:
def first_fail_seq(g):
    f = g.loc[g['PassOrFail'] == 1, 'shot_seq_in_run']
    return f.min() if len(f) else np.nan

RUNS = S.groupby('run_id').apply(lambda g: pd.Series({
    'product': g['product'].iloc[0],
    '시작': g['TimeStamp'].min(), '끝': g['TimeStamp'].max(),
    '길이(분)': round((g['TimeStamp'].max() - g['TimeStamp'].min()).total_seconds() / 60),
    '샷': len(g), '불량 샷': int(g['PassOrFail'].sum()),
    '불량률': round(g['PassOrFail'].mean(), 3),
    '짧은 정지': int(g['after_stop'].sum()),
    '첫 불량 위치(샷 순번)': first_fail_seq(g),
    '생산계획일': ', '.join(sorted(g['plan_date'].dt.strftime('%m-%d').unique())),
    '2차 범위': '✅' if g['in_secondary'].all() else ('❌' if not g['in_secondary'].any() else '일부'),
}), include_groups=False)
RUNS = RUNS.sort_values(['product', '시작'])
display(RUNS)

### 가동 구간과 불량 위치 그림

- 위: 시간축에 가동 구간(막대)과 불량 샷(빨간 점)
- 아래: 구간 시작 / 짧은 정지 후 몇 번째 샷에서 불량이 났는지 → 재시작 직후에 불량이 몰리는지 확인

In [ ]:
S2 = S[S['in_secondary']]
fig, axes = plt.subplots(len(PRODUCTS), 1, figsize=(12, 2.2 * len(PRODUCTS)))
for ax, p in zip(np.atleast_1d(axes), PRODUCTS):
    s = S2[S2['product'] == p]
    runs = s['run_id'].unique()
    for k, r in enumerate(runs):
        g = s[s['run_id'] == r]
        ax.plot(g['min_from_run_start'], np.full(len(g), k), color=LIGHT, lw=6, solid_capstyle='butt')
        st = g[g['after_stop']]
        ax.scatter(st['min_from_run_start'], np.full(len(st), k), marker='|', s=200, color='#52514e', zorder=3)
        f = g[g['PassOrFail'] == 1]
        ax.scatter(f['min_from_run_start'], np.full(len(f), k), s=22, color=RED, zorder=4)
    ax.set_yticks(range(len(runs)))
    ax.set_yticklabels([f"{r}  ({s[s['run_id'] == r]['TimeStamp'].min():%m-%d %H:%M})" for r in runs], fontsize=8)
    ax.set_xlabel('구간 시작 후 경과 시간 (분)', fontsize=8)
    ax.set_title(f'{p.upper()} · 가동 구간 (빨간 점 = 불량 샷, 세로선 = 짧은 정지)', loc='left', fontsize=10)
    ax.invert_yaxis()
fig.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6), sharey=True)
bins = [0, 10, 20, 30, 60, 120, 1000]
labels = ['1~10', '11~20', '21~30', '31~60', '61~120', '121~']
for ax, p in zip(axes, PRODUCTS):
    s = S2[S2['product'] == p].copy()
    s['구간'] = pd.cut(s['shots_since_stop'], bins=bins, labels=labels)
    t = s.groupby('구간', observed=False)['PassOrFail'].agg(['size', 'sum'])
    rate = (t['sum'] / t['size']).fillna(0)
    ax.bar(range(len(t)), rate, color=BLUE, width=0.6)
    for i, (n, f) in enumerate(zip(t['size'], t['sum'])):
        ax.text(i, rate.iloc[i], f'{int(f)}/{int(n)}', ha='center', va='bottom', fontsize=8)
    ax.set_xticks(range(len(t))); ax.set_xticklabels(labels, fontsize=8)
    ax.set_xlabel('구간 시작·짧은 정지 후 몇 번째 샷인지'); ax.set_title(f'{p.upper()} · 불량률 (불량/샷)', loc='left', fontsize=10)
axes[0].set_ylabel('불량률')
fig.tight_layout(); plt.show()

## 5. 저장

In [ ]:
COLS = (['shot_key', 'product', 'run_id', 'run_no', 'TimeStamp', 'plan_date', 'in_secondary',
         'shot_seq_in_run', 'min_from_run_start', 'prev_gap_sec', 'after_stop', 'shots_since_stop',
         'PassOrFail', 'fail_pattern', 'LH_fail', 'RH_fail', 'Reason_LH', 'Reason_RH',
         'n_rows', 'cavities', 'sec_index_LH', 'sec_index_RH'] + FEATURES + RAW)
S[COLS].to_csv(OUT_PATH, index=False, encoding='utf-8-sig')
RUNS.to_csv(os.path.join(os.path.dirname(OUT_PATH), 'runs_labeled.csv'), encoding='utf-8-sig')
print('저장:', os.path.abspath(OUT_PATH), S[COLS].shape)
print('저장:', os.path.abspath(os.path.join(os.path.dirname(OUT_PATH), 'runs_labeled.csv')))

## 사용법

```python
S = pd.read_csv('./linked/shots_labeled.csv', parse_dates=['TimeStamp', 'plan_date'])
S = S[S['in_secondary']]                 # 대회 2차 범위만 (기본)
cn7 = S[S['product'] == 'cn7']
X = cn7[FEATURES]                        # 2차 표준화 값 (또는 RAW로 원래 단위)
y = cn7['PassOrFail']                    # 샷 라벨
groups = cn7['run_id']                   # 분할·일자 내 보정은 날짜 대신 run 단위로
```

**주의**
- `Reason_*`, `LH_fail`, `RH_fail`, `fail_pattern`은 **모델 입력 금지** (검사 후 정보)
- `in_secondary=False` (CN7 10-27)는 대회 2차 범위 밖 → 쓰려면 사용 이유 설명 필요
- 02·03 노트북의 "일자 내" 보정은 `ts_date` 대신 `run_id`로 바꿔야 함